# Model Deployment using vLLM on Modal

## 📚 Learning Objectives

By the end of this notebook, you will understand:
1. What vLLM is and why it's useful for serving LLMs
2. What Modal is and how it simplifies deployment
3. How to deploy a model using vLLM on Modal
4. Best practices for production deployment
5. Performance optimization techniques


## 🎯 Understanding the Components

### What is vLLM?

**vLLM** (very Large Language Model) is a high-throughput and memory-efficient inference engine for LLMs.

**Key Features:**
- **PagedAttention**: Efficient memory management inspired by OS virtual memory
- **Continuous Batching**: Processes requests as they arrive for better throughput
- **Fast Model Execution**: Optimized CUDA kernels
- **Quantization Support**: FP16, INT8, INT4 for reduced memory
- **OpenAI-Compatible API**: Easy integration with existing tools

**Performance Benefits:**
- Up to 24x higher throughput than HuggingFace Transformers
- Better GPU utilization (80-90% vs 30-40%)
- Lower latency for concurrent requests

### What is Modal?

**Modal** is a serverless platform for running code in the cloud.

**Key Features:**
- **Serverless GPU Access**: Pay only for what you use
- **Container Management**: Automatic packaging and deployment
- **Auto-scaling**: Scale from 0 to many replicas
- **Simple Python API**: Define infrastructure as code
- **Built-in Secrets**: Secure credential management

**Why Modal + vLLM?**
- vLLM provides fast inference
- Modal handles infrastructure, scaling, and deployment
- Together: production-ready LLM serving in minutes





## 🧪Exercises

### Exercise 1: Deploy Your First Model
1. Choose a small model from HuggingFace (e.g., TinyLlama)
2. Create a basic deployment script
3. Deploy it to Modal
4. Test it with a few prompts

### Exercise 2: Add an API Endpoint
1. Modify your deployment to include a web endpoint
2. Add request/response validation
3. Test with curl or requests library

### Exercise 3: Implement Monitoring
1. Add latency tracking
2. Track tokens per second
3. Create a stats endpoint
4. Log errors and exceptions

### Exercise 4: Try Quantization
1. Find a quantized version of a 7B model
2. Deploy it on a smaller GPU (A10G)
3. Compare memory usage and speed with FP16 version

### Exercise 5 (small project): Build a Chat Application
1. Implement conversation history management
2. Add streaming responses
3. Create a simple web UI (optional)
4. Deploy as a persistent service

## 🔧 Part 1: Setup and Prerequisites

### Installation

In [1]:
# Install Modal CLI
!pip install modal -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.8/802.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.3/467.3 kB 11.0 MB/s eta 0:00:00


In [2]:
!pip install -U modal

### Authentication

You'll need to authenticate with Modal:

In [18]:
# Run this in your colab terminal (not in notebook)
!modal token new

# This will open a browser window for authentication
# Follow the prompts to create an account or sign in

Was not able to launch web browser
Please go to this URL manually and complete the flow:

]8;id=312857;https://modal.com/token-flow/tf-eQ8G2qnASNK2N7XtjMUdmp\https://modal.com/token-flow/tf-eQ8G2qnASNK2N7XtjMUdmp]8;;\

⠋ Waiting for authentication in the web browser
⠦ Waiting for token flow to complete...


### HuggingFace Token (Optional)

For gated models such as Llama or Mistral, you normally need a Hugging Face access token. However, for this challenge, we will not work with gated models, so no token is required. SO SKIP THIS PART !!!

In [20]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [22]:
# Set HuggingFace token as Modal secret
# Run in terminal:
!modal secret create huggingface HF_TOKEN=HF_TOKEN

╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ Secret 'huggingface' already exists in environment 'main'                    │
╰──────────────────────────────────────────────────────────────────────────────╯





## 🚀 Part 2: Basic Deployment Example

Let's deploy a small model first to understand the workflow.

In [23]:
%%writefile vllm_basic.py
import modal

app = modal.App("vllm-basic-example")

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

vllm_image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install(
        "vllm==0.9.2",
        "transformers==4.53.2",
        "accelerate",
    )
)

@app.cls(
    image=vllm_image,
    gpu="T4",
    scaledown_window=300,
)
class VLLMModel:
    @modal.enter()
    def load_model(self):
        from vllm import LLM

        self.llm = LLM(
            model=MODEL_NAME,
            dtype="float16",
            max_model_len=1024,
            gpu_memory_utilization=0.80,
            trust_remote_code=True,
        )

    @modal.method()
    def generate(self, prompt: str, max_tokens: int = 128) -> str:
        from vllm import SamplingParams

        params = SamplingParams(
            temperature=0.7,
            top_p=0.95,
            max_tokens=max_tokens,
        )

        outputs = self.llm.generate([prompt], params)
        return outputs[0].outputs[0].text


@app.local_entrypoint()
def main():
    model = VLLMModel()

    prompt = "Explain machine learning in simple terms."
    result = model.generate.remote(prompt)

    print("Prompt:", prompt)
    print("Generated:", result)

Writing vllm_basic.py


### Running this on terminal

```bash
# Deploy and run
modal run vllm_basic.py

# Deploy as a persistent service
modal deploy vllm_basic.py
```

## 🌐 Part 3: Web API Deployment

Let's create a production-ready API endpoint.

In [24]:
%%writefile vllm_api.py
import modal
from pydantic import BaseModel

app = modal.App("vllm-api")

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

vllm_image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install(
        "vllm==0.9.2",
        "transformers==4.53.2",
        "fastapi[standard]==0.115.0",
    )
)

class CompletionRequest(BaseModel):
    prompt: str
    max_tokens: int = 256
    temperature: float = 0.7
    top_p: float = 0.95

class CompletionResponse(BaseModel):
    text: str
    tokens_used: int
    model: str

@app.cls(
    image=vllm_image,
    gpu="T4",
    scaledown_window=600,
)
@modal.concurrent(max_inputs=10)
class VLLMServer:
    @modal.enter()
    def startup(self):
        from vllm import LLM

        self.llm = LLM(
            model=MODEL_NAME,
            dtype="float16",
            max_model_len=1024,
            gpu_memory_utilization=0.80,
            trust_remote_code=True,
        )

    @modal.method()
    def generate(self, request: CompletionRequest) -> CompletionResponse:
        from vllm import SamplingParams

        params = SamplingParams(
            temperature=request.temperature,
            top_p=request.top_p,
            max_tokens=request.max_tokens,
        )

        outputs = self.llm.generate([request.prompt], params)
        output = outputs[0].outputs[0]

        return CompletionResponse(
            text=output.text,
            tokens_used=len(output.token_ids),
            model=MODEL_NAME,
        )

    @modal.fastapi_endpoint(method="POST")
    def api_generate(self, request: CompletionRequest) -> CompletionResponse:
        return self.generate.local(request)

@app.local_entrypoint()
def test_api():
    server = VLLMServer()

    request = CompletionRequest(
        prompt="Write a haiku about AI:",
        max_tokens=100,
        temperature=0.8,
    )

    response = server.generate.remote(request)
    print("Generated:", response.text)
    print("Tokens used:", response.tokens_used)

Writing vllm_api.py


### Running this on terminal

```bash
# Deploy and run
modal run vllm_api.py

# Deploy as a persistent service
modal serve vllm_api.py
```

### Testing the API

After deployment, you can call it via HTTP:

In [27]:
import requests
import json

# Replace with your Modal endpoint URL
API_URL = "https://hbouanane--vllm-api-vllmserver-api-generate-dev.modal.run"

def call_api(prompt: str, max_tokens: int = 256):
    """Call the deployed vLLM API"""
    payload = {
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": 0.7,
        "top_p": 0.95,
    }

    response = requests.post(API_URL, json=payload)
    response.raise_for_status()

    result = response.json()
    print(f"Generated text: {result['text']}")
    print(f"Tokens used: {result['tokens_used']}")
    return result

# Example usage
call_api("Explain quantum computing in one sentence:")

Generated text:  Quantum computers are designed to solve problems that would be impossible for classical computers, but they require specialized hardware to achieve this.
Tokens used: 27


{'text': ' Quantum computers are designed to solve problems that would be impossible for classical computers, but they require specialized hardware to achieve this.',
 'tokens_used': 27,
 'model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'}



## 🎨 Part 4: Quantization for Memory Efficiency

Use quantization to run larger models on smaller GPUs.

In [28]:
%%writefile vllm_quantized.py
"""
Deploy quantized model with vLLM on Modal
"""

import modal

app = modal.App("vllm-quantized")

MODEL_NAME = "TheBloke/Llama-2-7B-Chat-AWQ"

vllm_image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install(
        "vllm==0.9.2",
        "transformers==4.53.2",
    )
)

@app.cls(
    image=vllm_image,
    gpu="T4",
    scaledown_window=600,
)
class QuantizedVLLM:
    @modal.enter()
    def load(self):
        from vllm import LLM

        self.llm = LLM(
            model=MODEL_NAME,
            quantization="awq",
            dtype="float16",
            max_model_len=1024,
            gpu_memory_utilization=0.80,
            trust_remote_code=True,
        )

        print("Quantized model loaded!")

    @modal.method()
    def generate(self, prompt: str, max_tokens: int = 256) -> str:
        from vllm import SamplingParams

        sampling_params = SamplingParams(
            temperature=0.7,
            top_p=0.95,
            max_tokens=max_tokens,
        )

        outputs = self.llm.generate([prompt], sampling_params)
        return outputs[0].outputs[0].text


@app.local_entrypoint()
def main():
    model = QuantizedVLLM()

    prompt = "Explain quantization in simple terms."
    result = model.generate.remote(prompt, max_tokens=150)

    print("Prompt:", prompt)
    print("Output:", result)

Writing vllm_quantized.py


In [ ]:
#Run this on terminal
#modal run vllm_quantized.py

### Quantization Methods Comparison

| Method | Bits | Memory Reduction | Speed | Quality |
|--------|------|------------------|-------|----------|
| FP16   | 16   | Baseline         | Fast  | Best |
| INT8   | 8    | 50%              | Fast  | Good |
| AWQ    | 4    | 75%              | Fast  | Good |
| GPTQ   | 4    | 75%              | Fast  | Good |
| SqueezeLLM | 3-4 | 75-80%      | Medium | Fair |

## 📊 Part 5: Monitoring and Observability

In [29]:
%%writefile vllm_final_app.py
"""
Add monitoring and metrics to your vLLM deployment on Modal
"""

import modal
from datetime import datetime
from typing import Dict
import time

app = modal.App("vllm-monitoring")

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

vllm_image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install(
        "vllm==0.9.2",
        "transformers==4.53.2",
    )
)

@app.cls(
    image=vllm_image,
    gpu="T4",
    scaledown_window=600,
)
class MonitoredVLLM:
    @modal.enter()
    def load(self):
        from vllm import LLM

        self.llm = LLM(
            model=MODEL_NAME,
            dtype="float16",
            max_model_len=1024,
            gpu_memory_utilization=0.80,
            trust_remote_code=True,
        )

        self.request_count = 0
        self.total_tokens = 0
        self.total_latency = 0.0

    @modal.method()
    def generate(self, prompt: str, max_tokens: int = 256) -> Dict:
        from vllm import SamplingParams

        start_time = time.time()

        sampling_params = SamplingParams(
            temperature=0.7,
            top_p=0.95,
            max_tokens=max_tokens,
        )

        outputs = self.llm.generate([prompt], sampling_params)
        output = outputs[0].outputs[0]

        generated_text = output.text
        tokens_generated = len(output.token_ids)

        latency = time.time() - start_time

        self.request_count += 1
        self.total_tokens += tokens_generated
        self.total_latency += latency

        tokens_per_second = tokens_generated / latency if latency > 0 else 0

        return {
            "text": generated_text,
            "metrics": {
                "tokens_generated": tokens_generated,
                "latency_seconds": round(latency, 3),
                "tokens_per_second": round(tokens_per_second, 2),
                "timestamp": datetime.now().isoformat(),
            },
        }

    @modal.method()
    def get_stats(self) -> Dict:
        avg_latency = (
            self.total_latency / self.request_count
            if self.request_count > 0
            else 0
        )

        avg_tokens_per_request = (
            self.total_tokens / self.request_count
            if self.request_count > 0
            else 0
        )

        return {
            "total_requests": self.request_count,
            "total_tokens_generated": self.total_tokens,
            "average_latency_seconds": round(avg_latency, 3),
            "average_tokens_per_request": round(avg_tokens_per_request, 2),
        }


@app.local_entrypoint()
def main():
    model = MonitoredVLLM()

    prompts = [
        "What is Python?",
        "Explain machine learning:",
        "What is cloud computing?",
    ]

    for prompt in prompts:
        result = model.generate.remote(prompt, max_tokens=100)

        print("\nPrompt:", prompt)
        print("Generated:", result["text"])
        print("Metrics:", result["metrics"])

    stats = model.get_stats.remote()

    print("\n=== Aggregate Statistics ===")
    for key, value in stats.items():
        print(f"{key}: {value}")

Writing vllm_final_app.py


In [ ]:
#modal run vllm_final_app.py
#modal deploy vllm_final_app.py

## 🔐 Part 6: Production Best Practices

### 1. Secret Management

In [ ]:
# Store secrets in Modal
# Terminal commands:

# HuggingFace token for gated models
# modal secret create huggingface HF_TOKEN=hf_xxxxxxxxxxxxx

# API keys for authentication
# modal secret create api-keys API_KEY=your_secret_key

# Use in code:
# @app.cls(
#     secrets=[modal.Secret.from_name("huggingface"), modal.Secret.from_name("api-keys")]
# )

### 2. Cost Optimization Tips




#### 🎯 Recommended Free Colab Setup

| Component | Recommended Choice | Why |
|---|---|---|
| **GPU** | T4 16GB | Usually available on Free Colab |
| **Model Size** | 1B–3B models | Fits comfortably in limited VRAM |
| **Precision** | FP16 | Reduces memory usage |
| **Context Length** | 1024 tokens | Faster and cheaper inference |
| **Batch Size** | 1 | Prevents GPU memory crashes |



#### ✅ Best Models for Free Colab

| Model | Size | Colab Free Compatibility |
|---|---:|---|
| TinyLlama 1.1B | 1.1B | ✅ Excellent |
| Microsoft Phi-2 | 2.7B | ✅ Good |
| Gemma 2B | 2B | ✅ Good |
| Quantized 7B models | 7B | ⚠️ Possible with quantization |



#### 🚀 Cost-Saving Strategies

##### 1. Use Small Models First

Start with lightweight models before experimenting with larger LLMs.

Benefits:
- lower VRAM usage
- faster inference
- fewer runtime crashes
- reduced startup time



##### 2. Reduce Context Length

Smaller context windows help:
- reduce GPU memory consumption
- improve latency
- increase throughput
- avoid CUDA out-of-memory errors



##### 3. Use Quantized Models

Quantization allows larger models to run on limited hardware.

Common quantization techniques:
- 4-bit
- 8-bit
- AWQ
- GPTQ
- GGUF

Benefits:
- lower VRAM usage
- faster loading
- ability to run larger models on T4 GPUs



##### 4. Keep Prompts Short

Shorter prompts:
- reduce inference cost
- improve response speed
- increase tokens/sec
- reduce memory pressure



##### 5. Clear GPU Memory Regularly

Cleaning unused GPU memory helps:
- prevent VRAM fragmentation
- avoid runtime crashes
- stabilize long notebook sessions


##### 6. Save Models to Google Drive

Benefits:
- avoid repeated downloads
- faster notebook startup
- reduced bandwidth usage
- better workflow continuity



#### ⚠️ Free Colab Limitations

| Limitation | Impact |
|---|---|
| Runtime disconnects | Not suitable for production |
| Random GPU allocation | T4 is not always guaranteed |
| Limited VRAM | Large models may fail |
| Temporary storage | Files disappear after runtime reset |
| No persistent APIs | Services stop after session ends |


#### 🧠 Key Takeaway

> Free Google Colab is ideal for:
>
> - learning
> - experimentation
> - prototyping
> - demos
>
> For production deployments and persistent APIs, use dedicated GPU platforms such as Modal, RunPod, or cloud GPU instances.

## 📚 Part 7: Use Case -*texte en italique* Building a Production-Ready LLM Inference Platform with vLLM, Modal, FastAPI, Monitoring, and Interactive Web UI

######*This use case demonstrates the deployment of a production-ready conversational AI application using vLLM and Modal. The application serves a Hugging Face language model through scalable API endpoints, supports multi-turn chat interactions, tracks inference metrics such as latency and tokens per second, and provides an interactive web-based UI for real-time conversations.*

TRY REPLACING younes IN 'stats_url' AND 'chat_url' IN THE CODE WITH YOUR OWN USERNAME.



In [31]:
%%writefile vllm_final_app.py

import modal
import time
from datetime import datetime
from typing import List, Dict
from pydantic import BaseModel

app = modal.App("vllm-final-app")

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install(
        "vllm==0.9.2",
        "transformers==4.53.2",
        "fastapi[standard]==0.115.0",
    )
)


class Message(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    messages: List[Message]
    max_tokens: int = 150
    temperature: float = 0.7
    top_p: float = 0.95


@app.cls(
    image=image,
    gpu="T4",
    scaledown_window=600,
)
@modal.concurrent(max_inputs=10)
class VLLMChatApp:

    @modal.enter()
    def load_model(self):
        from vllm import LLM

        self.llm = LLM(
            model=MODEL_NAME,
            dtype="float16",
            max_model_len=1024,
            gpu_memory_utilization=0.80,
            trust_remote_code=True,
        )

        self.total_requests = 0
        self.total_tokens = 0
        self.total_latency = 0.0
        self.errors = 0

    def format_chat(self, messages: List[Message]) -> str:
        prompt = ""

        for msg in messages:
            if msg.role == "system":
                prompt += f"<|system|>\n{msg.content}</s>\n"
            elif msg.role == "user":
                prompt += f"<|user|>\n{msg.content}</s>\n"
            elif msg.role == "assistant":
                prompt += f"<|assistant|>\n{msg.content}</s>\n"

        prompt += "<|assistant|>\n"
        return prompt

    @modal.method()
    def generate(self, request: ChatRequest) -> Dict:
        from vllm import SamplingParams

        try:
            start = time.time()

            prompt = self.format_chat(request.messages)

            sampling_params = SamplingParams(
                temperature=request.temperature,
                top_p=request.top_p,
                max_tokens=request.max_tokens,
                stop=["</s>", "<|user|>", "<|system|>"],
            )

            outputs = self.llm.generate([prompt], sampling_params)
            output = outputs[0].outputs[0]

            latency = time.time() - start
            tokens_generated = len(output.token_ids)
            tokens_per_second = tokens_generated / latency if latency > 0 else 0

            self.total_requests += 1
            self.total_tokens += tokens_generated
            self.total_latency += latency

            return {
                "text": output.text.strip(),
                "tokens_generated": tokens_generated,
                "latency_seconds": round(latency, 3),
                "tokens_per_second": round(tokens_per_second, 2),
                "model": MODEL_NAME,
                "timestamp": datetime.now().isoformat(),
            }

        except Exception as e:
            self.errors += 1
            return {
                "error": str(e),
                "model": MODEL_NAME,
                "timestamp": datetime.now().isoformat(),
            }

    @modal.method()
    def stats(self) -> Dict:
        avg_latency = self.total_latency / self.total_requests if self.total_requests else 0
        avg_tokens = self.total_tokens / self.total_requests if self.total_requests else 0

        return {
            "model": MODEL_NAME,
            "total_requests": self.total_requests,
            "total_tokens_generated": self.total_tokens,
            "average_latency_seconds": round(avg_latency, 3),
            "average_tokens_per_request": round(avg_tokens, 2),
            "errors": self.errors,
        }

    @modal.fastapi_endpoint(method="POST")
    def chat(self, request: ChatRequest) -> Dict:
        return self.generate.local(request)

    @modal.fastapi_endpoint(method="GET")
    def get_stats(self) -> Dict:
        return self.stats.local()

    @modal.fastapi_endpoint(method="GET")
    def ui(self):
        from fastapi.responses import HTMLResponse

        chat_url = "https://hbouanane--vllm-final-app-vllmchatapp-chat.modal.run" #REPLACE WITH YOUR OWN URL
        stats_url = "https://hbouanane--vllm-final-app-vllmchatapp-get-stats.modal.run" #REPLACE WITH YOUR OWN URL

        html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>vLLM Chat UI</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            max-width: 900px;
            margin: 40px auto;
            padding: 20px;
            background: #f4f4f4;
        }}
        .container {{
            background: white;
            padding: 25px;
            border-radius: 12px;
            box-shadow: 0 2px 12px rgba(0,0,0,0.1);
        }}
        textarea {{
            width: 100%;
            height: 140px;
            padding: 12px;
            font-size: 16px;
            box-sizing: border-box;
        }}
        button {{
            margin-top: 12px;
            padding: 12px 24px;
            font-size: 16px;
            cursor: pointer;
        }}
        #response {{
            margin-top: 20px;
            background: #efefef;
            padding: 15px;
            border-radius: 8px;
            white-space: pre-wrap;
            min-height: 120px;
        }}
        #stats {{
            margin-top: 20px;
            background: #dfefff;
            padding: 15px;
            border-radius: 8px;
            white-space: pre-wrap;
        }}
    </style>
</head>

<body>
    <div class="container">
        <h1>vLLM Chat UI</h1>

        <textarea id="prompt" placeholder="Ask something..."></textarea>
        <br>

        <button onclick="sendMessage()">Send</button>
        <button onclick="loadStats()">Load Stats</button>

        <div id="response"></div>
        <div id="stats"></div>
    </div>

<script>
async function sendMessage() {{

    const prompt = document.getElementById("prompt").value;
    const responseDiv = document.getElementById("response");

    responseDiv.innerText = "Generating...";

    const res = await fetch("{chat_url}", {{
        method: "POST",
        headers: {{
            "Content-Type": "application/json"
        }},
        body: JSON.stringify({{
            messages: [
                {{
                    role: "system",
                    content: "You are a helpful AI assistant."
                }},
                {{
                    role: "user",
                    content: prompt
                }}
            ],
            max_tokens: 150,
            temperature: 0.7,
            top_p: 0.95
        }})
    }});

    const data = await res.json();

    if (data.text) {{
        responseDiv.innerText =
            data.text +
            "\\n\\n--- Metrics ---" +
            "\\nTokens: " + data.tokens_generated +
            "\\nLatency: " + data.latency_seconds + "s" +
            "\\nTokens/sec: " + data.tokens_per_second +
            "\\nModel: " + data.model;
    }} else {{
        responseDiv.innerText = "Error: " + JSON.stringify(data);
    }}
}}

async function loadStats() {{

    const statsDiv = document.getElementById("stats");

    statsDiv.innerText = "Loading stats...";

    const res = await fetch("{stats_url}");
    const data = await res.json();

    statsDiv.innerText =
        "=== Aggregate Statistics ===\\n\\n" +
        "Model: " + data.model + "\\n" +
        "Total Requests: " + data.total_requests + "\\n" +
        "Total Tokens: " + data.total_tokens_generated + "\\n" +
        "Average Latency: " + data.average_latency_seconds + "s\\n" +
        "Average Tokens/Request: " + data.average_tokens_per_request + "\\n" +
        "Errors: " + data.errors;
}}
</script>

</body>
</html>
        """

        return HTMLResponse(html)


@app.local_entrypoint()
def main():

    model = VLLMChatApp()

    request = ChatRequest(
        messages=[
            Message(role="system", content="You are a helpful AI assistant."),
            Message(role="user", content="Explain machine learning simply."),
        ],
        max_tokens=120,
        temperature=0.7,
    )

    response = model.generate.remote(request)

    print("\\nChat response:")
    print(response)

    stats = model.stats.remote()

    print("\\nStats:")
    print(stats)

Overwriting vllm_final_app.py


In [ ]:
#modal deploy vllm_final_app.py

You can open the deployment URL displayed in the terminal after clicking on "View Deployment". You will see 3 different URLs: one for the UI, one for the chat API, and one for the statistics endpoint. Open just the UI URL and wait a few moments for the model to initialize. After that, the interface will appear and you can start chatting with the model.



## 📖 Part 11: Additional Resources

### Documentation
- [vLLM Documentation](https://docs.vllm.ai/)
- [Modal Documentation](https://modal.com/docs)
- [vLLM GitHub](https://github.com/vllm-project/vllm)

### Model Sources
- [HuggingFace Hub](https://huggingface.co/models)
- [Quantized Models by TheBloke](https://huggingface.co/TheBloke)


### Performance Benchmarks
- [vLLM Performance](https://blog.vllm.ai/2023/06/20/vllm.html)
- [Modal GPU Pricing](https://modal.com/pricing)

## 🎓 Summary

You've learned:

✅ **What vLLM is** and why it's the fastest way to serve LLMs  
✅ **What Modal is** and how it simplifies cloud deployment  
✅ **Basic deployment** of models with vLLM on Modal  
✅ **API endpoints** for production use  
✅ **Batching and streaming** for efficient inference  
✅ **Quantization** to run bigger models on smaller GPUs  
✅ **Monitoring** to track performance  
✅ **Best practices** for production deployments  

### Next Steps

1. **Experiment** with different models and configurations
2. **Optimize** for your specific use case (latency vs throughput)
3. **Scale** by adjusting concurrency and GPU settings
4. **Monitor** costs and performance in production
5. **Iterate** based on real user feedback

Happy deploying! 🚀